In [ ]:
import sys
sys.path.append('..')

import torch
import torchvision
from micro_diffusion.datasets.latents_loader import build_streaming_latents_dataloader

In [ ]:
micro_path = './datadir/diffdb/wds_latents_sdxl1_dfnclipH14/0'
restored_path = './outputs/restored_images/ambient-o/restore_diffdb/sigma-tn-2.0/guidance-scale-3.5_sampling-steps-30/0'
num_images_per_row = 8

In [ ]:
micro_loader = build_streaming_latents_dataloader(
    micro_path,
    image_size=512,
    batch_size=num_images_per_row,
    cap_seq_size=77,
    cap_emb_dim=1024,
    shuffle=False,
    drop_last=False
)
micro_dataset = micro_loader.dataset
micro_iter = iter(micro_loader)

In [ ]:
restored_loader = build_streaming_latents_dataloader(
    restored_path,
    image_size=512,
    batch_size=num_images_per_row,
    cap_seq_size=77,
    cap_emb_dim=1024,
    shuffle=False,
    drop_last=False
)
restored_dataset = restored_loader.dataset
restored_iter = iter(restored_loader)

In [ ]:
from micro_diffusion.models.model import create_latent_diffusion
from huggingface_hub import hf_hub_download
import torch

# Init model
params = {
    'latent_res': 64,
    'in_channels': 4,
    'pos_interp_scale': 2.0,
    'dtype': 'float32',
}
model = create_latent_diffusion(**params)
model.vae = model.vae.to('cuda')

In [ ]:
@torch.no_grad()
def latents_to_image(latents):
    latents = 1 / model.latent_scale * latents
    torch_dtype = torch.float32
    image = model.vae.decode(latents.to(torch_dtype)).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.float().detach()
    return image

In [ ]:
# micro_batch = next(micro_iter)
# restored_batch = next(restored_iter)
generator = torch.Generator()
generator.manual_seed(42)
idxs = torch.randint(0, len(micro_dataset)+1, (num_images_per_row,), generator=generator)

micro_batch = torch.stack([micro_dataset[i.item()]['image_latents'] for i in idxs])
restored_batch = torch.stack([restored_dataset[i.item()]['image_latents'] for i in idxs])
print([micro_dataset[i.item()]['caption'] for i in idxs])

micro_images = latents_to_image(micro_batch.to('cuda'))
restored_images = latents_to_image(restored_batch.to('cuda'))
 
images = torch.concat((micro_images, restored_images), dim=0)

In [ ]:
torchvision.transforms.ToPILImage()(torchvision.utils.make_grid(images, nrow=num_images_per_row, padding=2, normalize=False))